# 🤖 Machine Learning - Segmentação de Clientes (E-Commerce & Varejo) com K-Means

**Autor:** Victor Oliveira  
**Pós-Graduação:** Ciência de Dados e Inteligência Artificial (Universidade São Judas)  
**Stack:** Python, `Scikit-Learn`, `Pandas`, `NumPy`, `Matplotlib`, `Seaborn`  

---

## 🎯 Objetivos de Negócio
1. **Segmentação de Consumidores**: Cruzar a **Renda Anual ($k$)** com o **Score de Gastos (1 a 100)** para identificar perfis de comportamento.
2. **Determinar o Número de Personas (K)**: Aplicação da métrica de **Inércia** e o **Método do Cotovelo (*Elbow Method*)**.
3. **Mapeamento de Centróides**: Identificar as coordenadas médias de cada perfil de cliente.
4. **Avaliação da Qualidade**: Cálculo e interpretação do **Score de Silhueta (*Silhouette Score*)**.

### 1. Importação de Bibliotecas e Carga dos Dados de Clientes

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Simulação de dados de clientes (Renda Anual vs Score de Gastos)
np.random.seed(42)
c1 = np.random.multivariate_normal([25, 20], [[15, 2], [2, 15]], 150)
c2 = np.random.multivariate_normal([25, 80], [[15, 2], [2, 15]], 150)
c3 = np.random.multivariate_normal([55, 50], [[25, 2], [2, 25]], 300)
c4 = np.random.multivariate_normal([85, 20], [[20, 2], [2, 20]], 150)
c5 = np.random.multivariate_normal([85, 80], [[20, 2], [2, 20]], 150)

X = np.vstack([c1, c2, c3, c4, c5])
df = pd.DataFrame(X, columns=['Renda_Anual_kUSD', 'Score_Gastos_1_100'])
df.head()

### 2. Escolha do K Ideal: O Método do Cotovelo (*Elbow Method*)

In [2]:
inercias = []
k_range = range(1, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inercias.append(kmeans.inertia_)

# Plot do Método do Cotovelo
plt.figure(figsize=(9, 5))
plt.plot(k_range, inercias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (k)', fontsize=12)
plt.ylabel('Inércia', fontsize=12)
plt.title('Método do Cotovelo (Elbow Method)', fontsize=13, fontweight='bold')
plt.axvline(x=5, color='r', linestyle='--', label='K Ideal = 5')
plt.legend()
plt.show()

### 3. Mapeamento dos Centróides e Fronteiras dos Grupos de Consumo

In [3]:
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans_final.fit_predict(X)
centroides = kmeans_final.cluster_centers_

x_min, x_max = X[:, 0].min() - 5, X[:, 0].max() + 5
y_min, y_max = X[:, 1].min() - 5, X[:, 1].max() + 5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.5), np.arange(y_min, y_max, 0.5))
Z = kmeans_final.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(11, 6))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='tab10')
plt.scatter(X[:, 0], X[:, 1], c=df['Cluster'], cmap='tab10', s=15, alpha=0.6)
plt.scatter(centroides[:, 0], centroides[:, 1], c='red', marker='X', s=250, edgecolor='black', linewidth=1.5, label='Centróides')

for i, (cx, cy) in enumerate(centroides):
    plt.annotate(f'{cx:.0f}k$ | Score {cy:.0f}', xy=(cx, cy), xytext=(cx + 2, cy + 3),
                 bbox=dict(boxstyle='round,pad=0.3', fc='yellow', alpha=0.9),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=8.5, fontweight='bold')

plt.title('Segmentação Prática de Clientes (Renda Anual vs Score de Gastos)', fontsize=13, fontweight='bold')
plt.xlabel('Renda Anual do Cliente (k$)', fontsize=12)
plt.ylabel('Score de Gastos no App/E-Commerce (1 a 100)', fontsize=12)
plt.legend(loc='upper left')
plt.show()

### 4. Avaliação de Qualidade (Silhouette Score: 0.8037)

In [4]:
sil_score = silhouette_score(X, df['Cluster'])
print(f"Score de Silhueta (Silhouette Score): {sil_score:.4f}")